# Information Theory & Attention — Hands-On Notebook

Companion to: [Information Theory](../docs/01_foundations/information_theory.md), [Attention Mathematics](../docs/01_foundations/attention_math.md)

**What you'll build:**
- Entropy, cross-entropy, and KL divergence from scratch
- Scaled dot-product attention from scratch
- Multi-head attention from scratch
- Attention visualization from GPT-2 and BERT

## Section 1: Setup

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
sns.set_theme(style="whitegrid", context="notebook")

## Section 2: Entropy — Measuring Surprise

Entropy measures how unpredictable a distribution is. A fair coin (50/50) has maximum entropy. A biased coin (99/1) has low entropy — you're rarely surprised.

For discrete $P$, $H(P) = -\sum_x P(x)\log_2 P(x)$ (bits when using $\log_2$).

In [ ]:
def entropy_bits(probs, base=2):
    p = np.asarray(probs, dtype=float)
    p = np.clip(p, 1e-12, 1.0)
    return float(-np.sum(p * np.log(p) / np.log(base)))

fair = np.array([0.5, 0.5])
biased = np.array([0.9, 0.1])
uniform8 = np.ones(8) / 8.0

print("Fair coin H =", entropy_bits(fair), "bits (expect 1.0)")
print("Biased [0.9, 0.1] H ≈", round(entropy_bits(biased), 2), "bits (expect ~0.47)")
print("Uniform over 8 H =", entropy_bits(uniform8), "bits (expect 3.0)")

# Approximate English letter frequencies (lowercase + space); rough but standard teaching set
letters = "etaoinshrdlcumwfgypbvkjxqz "
freqs = np.array(
    [12.7, 9.1, 8.2, 7.5, 6.7, 6.3, 6.1, 6.0, 5.6, 4.3, 4.0, 2.8, 2.7, 2.7, 2.4, 2.3, 2.2, 2.0, 2.0, 1.9, 1.5, 1.0, 0.97, 0.15, 0.15, 0.074, 13.0]
)
P_english = freqs / freqs.sum()
print("English (approx) letter+space entropy H ≈", round(entropy_bits(P_english), 2), "bits")

In [ ]:
p_grid = np.linspace(1e-6, 1 - 1e-6, 500)
H_binary = [entropy_bits([p, 1 - p]) for p in p_grid]

plt.figure(figsize=(7, 4))
plt.plot(p_grid, H_binary, color="steelblue", lw=2, label=r"$H(p)$ for Bernoulli$(p)$")
plt.axvline(0.5, color="gray", ls="--", alpha=0.7, label="Maximum at $p=0.5$")
plt.xlabel("Probability $p$ of heads")
plt.ylabel("Entropy (bits)")
plt.title("Binary entropy: uncertainty vs. bias")
plt.legend(loc="best")
plt.tight_layout()
plt.show()
print("Peak H ≈", max(H_binary), "bits at p ≈", p_grid[np.argmax(H_binary)])

## Section 3: Cross-Entropy — Model vs Reality

Cross-entropy measures how surprised your **model** is by the **real** data. If your model perfectly matches reality, cross-entropy equals entropy. Otherwise, it's higher.

$H(P, Q) = -\sum_x P(x)\log Q(x)$ (using the same log base as entropy).

In [ ]:
P = torch.tensor([0.7, 0.2, 0.1], dtype=torch.float32)
Q1 = torch.tensor([0.6, 0.3, 0.1], dtype=torch.float32)
Q2 = torch.tensor([0.1, 0.1, 0.8], dtype=torch.float32)

def cross_entropy(P, Q, base="e"):
    Q = Q.clamp(min=1e-12)
    ce = -(P * torch.log(Q)).sum()
    if base == 2:
        ce = ce / math.log(2.0)
    return ce

ce1 = cross_entropy(P, Q1, base="e")
ce2 = cross_entropy(P, Q2, base="e")
print("H(P, Q1) [nats]:", ce1.item())
print("H(P, Q2) [nats]:", ce2.item())
print("Q2 has much higher cross-entropy — it is far more 'surprised' by the real distribution.")

In [ ]:
# Verify with PyTorch: soft-target cross_entropy (target = probability vector)
logits1 = torch.log(Q1.clamp(min=1e-12)).unsqueeze(0)
logits2 = torch.log(Q2.clamp(min=1e-12)).unsqueeze(0)
target = P.unsqueeze(0)
ce_fn = nn.CrossEntropyLoss(reduction="sum")
ce_torch1 = ce_fn(logits1, target)
ce_torch2 = ce_fn(logits2, target)
print("CrossEntropyLoss H(P,Q1):", ce_torch1.item())
print("CrossEntropyLoss H(P,Q2):", ce_torch2.item())
print("Matches manual:", torch.allclose(ce1, ce_torch1), torch.allclose(ce2, ce_torch2))

This is exactly the loss function used to train LLMs — minimize cross-entropy so the model is less surprised by real text (in practice, $P$ is a one-hot true next token and $Q$ is the predicted distribution).

## Section 4: KL Divergence — The Cost of Being Wrong

$\mathrm{KL}(P\|Q) = H(P, Q) - H(P)$. It's the **extra** surprise from using $Q$ instead of $P$. Always $\geq 0$.

In [ ]:
H_P = -(P * torch.log(P.clamp(min=1e-12))).sum()
kl1 = ce1 - H_P
kl2 = ce2 - H_P
print("KL(P||Q1) from definition:", kl1.item())
print("KL(P||Q2) from definition:", kl2.item())

# F.kl_div(input, target): input=log(Q), target=P gives sum P*(log P - log Q) = KL(P||Q)
kl1_t = F.kl_div(torch.log(Q1.clamp(min=1e-12)), P, reduction="sum")
kl2_t = F.kl_div(torch.log(Q2.clamp(min=1e-12)), P, reduction="sum")
print("F.kl_div KL(P||Q1):", kl1_t.item())
print("F.kl_div KL(P||Q2):", kl2_t.item())

In [ ]:
# KL is asymmetric: KL(P||Q) vs KL(Q||P)
kl_qp1 = F.kl_div(torch.log(P.clamp(min=1e-12)), Q1, reduction="sum")
print("KL(Q1||P):", kl_qp1.item(), " vs KL(P||Q1):", kl1_t.item())

p_vals = np.linspace(0.01, 0.99, 200)
kl_line = []
for p in p_vals:
    Pp = torch.tensor([p, 1 - p], dtype=torch.float32)
    Qh = torch.tensor([0.5, 0.5], dtype=torch.float32)
    kl_line.append(F.kl_div(torch.log(Qh.clamp(min=1e-12)), Pp, reduction="sum").item())

plt.figure(figsize=(7, 4))
plt.plot(p_vals, kl_line, color="darkorange", lw=2, label=r"$\mathrm{KL}(P\|Q)$ with $Q=(0.5,0.5)$")
plt.xlabel(r"$p$ where $P=[p, 1-p]$")
plt.ylabel("KL divergence (nats)")
plt.title("KL from a fair coin model to varying $P$")
plt.legend()
plt.tight_layout()
plt.show()

## Section 5: Perplexity — The Intuitive Metric

Perplexity $= \exp(H(P,Q))$ when cross-entropy is in nats, or $2^{H_{\mathrm{bits}}}$ in bits. A perplexity of 50 means the model is about as uncertain as if it were choosing uniformly among 50 equally likely words.

In [ ]:
def perplexity_from_ce_nats(ce_nats):
    return math.exp(ce_nats)

print("Perplexity for Q1:", perplexity_from_ce_nats(ce1.item()))
print("Perplexity for Q2:", perplexity_from_ce_nats(ce2.item()))

V = 10000
uniform = torch.ones(V) / V
ce_uniform = -(uniform * torch.log(uniform)).sum()
print("Cross-entropy of uniform over V=10000 [nats]:", ce_uniform.item())
print("Perplexity (uniform model):", math.exp(ce_uniform.item()), "≈ V")
assert abs(math.exp(ce_uniform.item()) - V) < 1e-3

## Section 6: Scaled Dot-Product Attention from Scratch

Attention answers: *When processing this token, which other tokens should I focus on?* Three steps: **score** (how relevant?), **normalize** (weights), **blend** (weighted sum).

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

seq_len, d_k = 4, 3
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)
out, w = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)
print("Attention weights (first position attends to all keys):\n", w[0].numpy())
print("Output[0,0,:] sample:", out[0, 0].numpy())

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(w[0].detach().numpy(), annot=True, fmt=".2f", cmap="viridis", square=True)
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.title("Attention weights (unnmasked demo)")
plt.tight_layout()
plt.show()

**Why divide by $\sqrt{d_k}$?** Without it, dot products grow large with dimension, softmax saturates toward one-hot, and gradients can vanish.

In [ ]:
d_large = 64
Qb = torch.randn(1, 4, d_large)
Kb = torch.randn(1, 4, d_large)
Vb = torch.randn(1, 4, d_large)
scores_raw = Qb @ Kb.transpose(-2, -1)
scores_scaled = scores_raw / math.sqrt(d_large)
w_raw = F.softmax(scores_raw, dim=-1)
w_scaled = F.softmax(scores_scaled, dim=-1)
print("Max weight without scaling:", w_raw.max().item())
print("Max weight with scaling:", w_scaled.max().item())
print("Entropy of avg row — raw:", -(w_raw[0] * (w_raw[0].clamp(min=1e-12).log())).sum(-1).mean().item())
print("Entropy of avg row — scaled:", -(w_scaled[0] * (w_scaled[0].clamp(min=1e-12).log())).sum(-1).mean().item())

## Section 7: Causal (Autoregressive) Masking

In GPT-style models, each position may only attend to previous positions — not to the future when predicting the next token.

In [ ]:
L = 5
d = 3
Qc = torch.randn(1, L, d)
Kc = torch.randn(1, L, d)
Vc = torch.randn(1, L, d)
causal = torch.tril(torch.ones(L, L)).unsqueeze(0)
out_u, w_u = scaled_dot_product_attention(Qc, Kc, Vc, mask=None)
out_m, w_m = scaled_dot_product_attention(Qc, Kc, Vc, mask=causal)
print("Unmasked weights row 3:", w_u[0, 3].numpy())
print("Masked weights row 3:", w_m[0, 3].numpy())

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(w_u[0].detach().numpy(), ax=ax[0], cmap="magma", square=True, cbar_kws={"label": "weight"})
ax[0].set_title("Unmasked (full attention)")
ax[0].set_xlabel("Key index")
ax[0].set_ylabel("Query index")
sns.heatmap(w_m[0].detach().numpy(), ax=ax[1], cmap="magma", square=True, cbar_kws={"label": "weight"})
ax[1].set_title("Causal (lower triangular)")
ax[1].set_xlabel("Key index")
ax[1].set_ylabel("Query index")
plt.tight_layout()
plt.show()

## Section 8: Multi-Head Attention from Scratch

Multiple heads learn different attention patterns (syntax, long-range dependencies, etc.).

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        b, t, _ = x.shape
        Q = self.W_q(x).view(b, t, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(b, t, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(b, t, self.num_heads, self.d_k).transpose(1, 2)
        if mask is not None:
            mask = mask.unsqueeze(1)
        out, weights = scaled_dot_product_attention(Q, K, V, mask=mask)
        out = out.transpose(1, 2).contiguous().view(b, t, self.d_model)
        return self.W_o(out), weights

mha = MultiHeadAttention(d_model=8, num_heads=2)
x = torch.randn(1, 6, 8)
y, attn = mha(x)
print("Output shape:", tuple(y.shape))
print("Attention weights shape (batch, heads, query, key):", tuple(attn.shape))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for h in range(2):
    sns.heatmap(attn[0, h].detach().numpy(), ax=axes[h], cmap="coolwarm", square=True)
    axes[h].set_title(f"Head {h}")
    axes[h].set_xlabel("Key position")
    axes[h].set_ylabel("Query position")
plt.suptitle("Multi-head attention weights")
plt.tight_layout()
plt.show()

## Section 9: Attention in GPT-2 (Real Model)

We load pretrained GPT-2 and inspect attention maps. Some heads attend locally, some to sentence start, etc.

In [ ]:
from transformers import GPT2Model, GPT2Tokenizer

sentence = "The cat sat on the mat because it was tired"
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2Model.from_pretrained("gpt2", output_attentions=True)
model.eval()
inputs = tokenizer(sentence, return_tensors="pt")
with torch.no_grad():
    out = model(**inputs)
attns = out.attentions
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("Tokens:", tokens)
print("Number of layers:", len(attns))

In [ ]:
def plot_gpt_heads(layer_idx, heads=(0, 1, 2)):
    A = attns[layer_idx][0]
    fig, axes = plt.subplots(1, len(heads), figsize=(5 * len(heads), 4))
    if len(heads) == 1:
        axes = [axes]
    for ax, h in zip(axes, heads):
        sns.heatmap(A[h].numpy(), xticklabels=tokens, yticklabels=tokens, cmap="viridis", ax=ax, cbar_kws={"label": "weight"})
        ax.set_title(f"GPT-2 layer {layer_idx}, head {h}")
        ax.set_xlabel("Key")
        ax.set_ylabel("Query")
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

plot_gpt_heads(0, heads=(0, 1, 2))
plot_gpt_heads(11, heads=(0, 1, 2))

Notice: some heads emphasize the previous token, others look at distant tokens or delimiters — heads specialize.

## Section 10: Attention in BERT (Bidirectional)

BERT sees the full sentence in both directions, so attention matrices are not lower-triangular.

In [ ]:
from transformers import BertModel, BertTokenizer

bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased", output_attentions=True)
bert.eval()
binp = bert_tok(sentence, return_tensors="pt")
with torch.no_grad():
    bout = bert(**binp)
battn = bout.attentions[-1][0, 0].numpy()
gattn = attns[-1][0, 0].numpy()
btoks = bert_tok.convert_ids_to_tokens(binp["input_ids"][0])

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(gattn, xticklabels=tokens, yticklabels=tokens, cmap="magma", ax=ax[0], cbar_kws={"label": "weight"})
ax[0].set_title("GPT-2 last layer, head 0 (causal)")
ax[0].set_xlabel("Key")
ax[0].set_ylabel("Query")
sns.heatmap(battn, xticklabels=btoks, yticklabels=btoks, cmap="magma", ax=ax[1], cbar_kws={"label": "weight"})
ax[1].set_title("BERT last layer, head 0 (bidirectional)")
ax[1].set_xlabel("Key")
ax[1].set_ylabel("Query")
for a in ax:
    plt.setp(a.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()
print("GPT-2 attention to future positions is ~0 (causal). BERT uses full context.")

## Section 11: Key Takeaways

- **Entropy** quantifies uncertainty; **cross-entropy** is how surprised your model is by data; **KL** is the extra cost of using $Q$ instead of $P$.
- **Perplexity** turns cross-entropy into an “effective vocabulary size” intuition.
- **Scaled dot-product attention** computes relevance scores, softmax weights, and a value mixture; **$1/\sqrt{d_k}$** keeps gradients healthy.
- **Causal masking** enforces autoregressive order; **multi-head** attention learns multiple relationship types.
- **GPT-2** shows lower-triangular (causal) patterns; **BERT** shows full bidirectional attention over the sequence.